# Lanzar simulaciones desde pp_reset

Flujo equivalente a `Lanzar_simulacion_RRAM.ipynb` pero el ciclo arranca en **PP_reset**.

```
init  →  generar_estados_sp_set  →  exec --start-from pp_reset  →  plot
```

La sección **3 · Generar estados sp_set** sustituye a la ejecución de PP_set + SP_set:
crea un `actual_state` percolante para cada simulación y lo guarda en
`Init_data/phase_state_{n_save}_sp_set.*`.

Logging: nivel global por env var `RRAM_LOG_LEVEL=DEBUG|INFO|WARNING`.
Cada subprocess escribe `logs/log_simulacion_{N}.log`.

In [21]:
%load_ext autoreload
%autoreload 2

import concurrent.futures
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from RRAM import simulation_config, Percolation, utils
from RRAM.init_simulation import load_simulation_config
from RRAM.persistence import save_phase_state

ruta_raiz = Path.cwd()
print('Ruta raíz del proyecto:', ruta_raiz)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Ruta raíz del proyecto: /Users/antonio_lopez_torres/Documents/GitHub/RRAM_Simulation


## 0 · Estructura de carpetas

In [22]:
def setup_project_structure():
    """Crea todas las carpetas necesarias antes de ejecutar las simulaciones."""
    for folder in ['Init_data', 'Results', 'Results/Figures', 'logs']:
        Path(folder).mkdir(parents=True, exist_ok=True)
        print(f'OK carpeta: {folder}/')

setup_project_structure()

# Limpiar resultados y logs previos
carpeta_results = ruta_raiz / 'Results'
if carpeta_results.exists():
    shutil.rmtree(carpeta_results)
carpeta_results.mkdir(parents=True, exist_ok=True)
(carpeta_results / 'Figures').mkdir(parents=True, exist_ok=True)

log_dir = ruta_raiz / 'logs'
log_dir.mkdir(exist_ok=True)
for f in log_dir.iterdir():
    if f.is_file():
        try:
            f.unlink()
        except Exception as e:
            print(f'No se pudo eliminar {f}: {e}')

print('Estructura lista.')

OK carpeta: Init_data/
OK carpeta: Results/
OK carpeta: Results/Figures/
OK carpeta: logs/
Estructura lista.


## 1 · Generación de parámetros (CSV → Init_data)

In [23]:
manager = simulation_config.ConfigManager()
carpeta_init = 'Init_data'

# ── Ajusta el barrido según tu experimento ───────────────────────────────
parametros_barrido = {
    "grosor_filamento": [
        [5]
    ],
    "num_trampas": [20],
}
# ─────────────────────────────────────────────────────────────────────────

manager.add_zip(zip_params=parametros_barrido)
manager.export_to_init_data(carpeta_init)

num_simulaciones = len(manager.simulations)
print(f"Generadas {num_simulaciones} configuraciones en {carpeta_init}/.")

Generadas 1 configuraciones en Init_data/.


## 2 · INIT — pre-generar estados iniciales

Genera `Init_data/init_state_{i}.npz` para cada simulación del CSV.

In [24]:
resultado = subprocess.run(
    [sys.executable, '-m', 'RRAM', 'init'],
    cwd=str(ruta_raiz),
    capture_output=True, text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print('STDERR:', resultado.stderr)
    raise RuntimeError('init falló')

## 3 · Generar estados sp_set

Para cada simulación construimos el dict `final_state_sp_set` que necesita `PP_reset`
y lo guardamos en `Init_data/phase_state_{n_save}_sp_set.*`.

El dispositivo se inicializa en **LRS**: se parte del `init_state_N.npz` y se añade
una franja de vacantes continua en la zona de cada filamento para garantizar la
percolación. El perfil de temperatura se fija uniformemente a `params.init_temp`.

In [25]:
# TODO FUNCION QUE MODIFICARÉ PARA EL ESTADO INICIAL CON UNA DENSIDAD DETERMINADA DE VACANTES, PERO CON LA GARANTÍA DE QUE HAY UN FILAMENTO PERCOLANTE DESDE EL INICIO. ESTA FUNCIÓN SE USARÁ PARA GENERAR EL ESTADO INICIAL DE SP_SET.

def crear_franja_percolante(
    state: np.ndarray,
    cf_ranges: list,
    cf_centros: list,
    grosor_filamento,
) -> np.ndarray:
    """Activa (True) una franja horizontal continua en la zona de cada filamento."""
    state = state.copy()
    for idx, (row_min, row_max) in enumerate(cf_ranges):
        centro = cf_centros[idx]
        if isinstance(grosor_filamento, (list, tuple, np.ndarray)):
            grosor = int(grosor_filamento[idx]) if idx < len(grosor_filamento) else int(grosor_filamento[-1])
        else:
            grosor = int(grosor_filamento)
        mitad   = grosor // 2
        fila_ini = max(row_min, centro - mitad)
        fila_fin = min(row_max + 1, centro + mitad + 1)
        state[fila_ini:fila_fin, :] = True
    return state


def generar_estado_sp_set(
    num_simulation: int,
    init_data_dir: Path = Path('Init_data'),
    tiempo_sp_set: float = 0.0,
) -> bool:
    """
    Construye y guarda el estado final de SP_set para una simulación concreta.

    Returns:
        True si el dispositivo percola al final; False en caso contrario.
    """
    cfg      = load_simulation_config(num_simulation, init_data_dir=init_data_dir)
    params   = cfg.params
    sim_ctes = cfg.sim_ctes
    n_save   = num_simulation + 1

    eje_x = params.y_size
    eje_y = params.x_size

    # Estado base: vacantes aleatorias del init_state
    init_path   = init_data_dir / f'init_state_{num_simulation}'
    actual_state = utils.cargar_estado(init_path).astype(bool)

    # Forzar filamento percolante
    actual_state = crear_franja_percolante(
        actual_state,
        cf_ranges=cfg.cf_ranges,
        cf_centros=cfg.cf_centros,
        grosor_filamento=sim_ctes.grosor_filamento,
    )

    percola = Percolation.is_path(actual_state.astype(int))

    # Temperatura uniforme a temperatura ambiente; forma (eje_x, eje_y + 2)
    Temperatura_final = np.full(
        (eje_x, eje_y + 2),
        fill_value=float(params.init_temp),
        dtype=np.float64,
    )

    final_state_sp_set = {
        'actual_state'      : actual_state,
        'sim_ctes'          : sim_ctes,
        'params'            : params,
        'Temperatura_final' : Temperatura_final,
        'percola'           : percola,
        'centros_calculados': list(cfg.cf_centros),
        'tiempo_sp_set'     : tiempo_sp_set,
    }

    save_phase_state(
        state_dict     = final_state_sp_set,
        phase_name     = 'sp_set',
        num_simulation = n_save,
        init_data_dir  = init_data_dir,
    )
    return percola

In [26]:
init_data_dir = Path('Init_data')
errores_percola = []

for num_sim in range(num_simulaciones):
    percola = generar_estado_sp_set(num_sim, init_data_dir=init_data_dir)
    estado  = '✓ percola' if percola else '⚠ NO percola'
    print(f'  sim {num_sim:3d} (n_save={num_sim+1:3d}): {estado}')
    if not percola:
        errores_percola.append(num_sim)

print(f'\nEstados sp_set generados: {num_simulaciones}')
if errores_percola:
    print(f'⚠  Simulaciones sin percolación: {errores_percola}')
    print('   Revisa el grosor_filamento o los rangos de CF para esas configuraciones.')
else:
    print('Todas las simulaciones percolan correctamente.')

  sim   0 (n_save=  1): ✓ percola

Estados sp_set generados: 1
Todas las simulaciones percolan correctamente.


## 4 · EXEC desde pp_reset — lanzamiento paralelo

Cada simulación es un subprocess `python -m RRAM exec <num> --start-from pp_reset`.
Lee el estado guardado en `Init_data/phase_state_{n_save}_sp_set.*` y ejecuta
PP_reset + SP_reset directamente.

In [ ]:
# ── Configuración de ejecución ────────────────────────────────────────────
num_filamentos  = 1
log_level       = os.environ.get('RRAM_LOG_LEVEL', 'INFO')
num_procesadores = max(int(0.2 * (os.cpu_count() or 10)), 1)
# ─────────────────────────────────────────────────────────────────────────

# Excluir simulaciones que no percolaron (PP_reset no tendría sentido)
sims_a_ejecutar = [n for n in range(num_simulaciones) if n not in errores_percola]
print(
    f'Lanzando {len(sims_a_ejecutar)} simulaciones desde pp_reset '
    f'con {num_procesadores} procesadores. RRAM_LOG_LEVEL={log_level}'
)


def ejecutar_desde_pp_reset(num_simulacion: int) -> int:
    env = os.environ.copy()
    env['RRAM_LOG_LEVEL'] = log_level

    cmd = [
        sys.executable, '-m', 'RRAM', 'exec',
        str(num_simulacion),
        '--num-filamentos', str(num_filamentos),
        '--start-from', 'pp_reset',
    ]
    print(f'  Iniciando exec sim={num_simulacion + 1} (desde pp_reset)')
    completed = subprocess.run(cmd, cwd=str(ruta_raiz), env=env)
    if completed.returncode != 0:
        print(f'  AVISO: sim {num_simulacion + 1} terminó con código {completed.returncode}')
    return completed.returncode


start_time = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=num_procesadores) as executor:
    list(executor.map(ejecutar_desde_pp_reset, sims_a_ejecutar))
elapsed_h = (time.time() - start_time) / 3600

print(f'\nTodas las simulaciones han terminado.')
print(f'Tiempo total: {elapsed_h:.2f} h ({elapsed_h * 3600:.1f} s)')

Lanzando 1 simulaciones desde pp_reset con 2 procesadores. RRAM_LOG_LEVEL=INFO
  Iniciando exec sim=1 (desde pp_reset)


In [ ]:
# Notificación WhatsApp (opcional)
import urllib.parse
import requests

def enviar_notificacion_whatsapp(n_sims, tiempo_total):
    telefono = "+34689866027"
    apikey   = "2486810"
    mensaje  = f"✅ Simulaciones RRAM (desde pp_reset) completadas. Procesadas: {n_sims} en {tiempo_total:.2f}h."
    url = f"https://api.callmebot.com/whatsapp.php?phone={telefono}&text={urllib.parse.quote(mensaje)}&apikey={apikey}"
    try:
        requests.get(url, timeout=10)
    except Exception as e:
        print(f'Notificación no enviada: {e}')

enviar_notificacion_whatsapp(len(sims_a_ejecutar), elapsed_h)

## 5 · PLOT — generar figuras (no re-ejecuta)

Lee los `Data_*.npz` + `sim_metadata.json` de disco. Funciona también en una
sesión nueva sin necesidad de repetir el ciclo.

In [ ]:
from RRAM.plot_results import plot_estados

# Estado de todas las sims, todas las fases

plot_estados(plot_types=["muro"], phases=["pp_reset"], sim_indices=list(range(1, 3)))
plot_estados(plot_types=["thermal"], phases=["pp_reset"], sim_indices=list(range(1, 3)))
plot_estados(plot_types=["state"], phases=["pp_reset"], sim_indices=list(range(1, 3)))
plot_estados(plot_types=["clean_state"], phases=["pp_reset"], sim_indices=list(range(1, 3)))


{1: {'T_final_pp_reset': 521.9542066710401},
 2: {'T_final_pp_reset': 521.9542066710401}}

## 6 · Extracción de datos

In [ ]:
# import json

# carpeta_resultados = Path('Results')


# def extraer_datos_simulaciones(directorio_base: Path):
#     datos_extraidos = []

#     for file_path in directorio_base.rglob('sim_metadata_*.json'):
#         with open(file_path, 'r', encoding='utf-8') as f:
#             try:
#                 data = json.load(f)
#             except json.JSONDecodeError:
#                 print(f'Error al leer el JSON (corrupto o vacío): {file_path}')
#                 continue

#         try:
#             atom_size_nm = data['params_dict']['atom_size'] * 1e9
#             grosor_1 = data['ctes_dict']['grosor_filamento'][0]
#             grosor_2 = data['ctes_dict']['grosor_filamento'][1]
#             diam_1   = (2 * grosor_1 + 1) * atom_size_nm
#             diam_2   = (2 * grosor_2 + 1) * atom_size_nm

#             v_crea_1 = v_crea_2 = np.nan
#             v_rot_1  = v_rot_2  = np.nan

#             for evento in data.get('creaciones_dict', {}).values():
#                 if evento['filamento'] == 1:
#                     v_crea_1 = evento['voltaje']
#                 elif evento['filamento'] == 2:
#                     v_crea_2 = evento['voltaje']

#             for evento in data.get('roturas_dict', {}).values():
#                 if evento['filamento'] == 1:
#                     v_rot_1 = evento['voltaje']
#                 elif evento['filamento'] == 2:
#                     v_rot_2 = evento['voltaje']

#             datos_extraidos.append([diam_1, diam_2, v_crea_1, v_crea_2, v_rot_1, v_rot_2])

#         except KeyError as e:
#             print(f'Saltando {file_path.name}: Falta la clave {e}')

#     if datos_extraidos:
#         header = (
#             'Diametro CF 1 (nm)\tDiametro CF 2 (nm)\t'
#             'Tension Creacion CF 1 (V)\tTension Creacion CF 2 (V)\t'
#             'Tension rotura CF 1 (V)\tTension rotura CF 2 (V)'
#         )
#         nombre_salida = 'resumen_tensiones_filamentos.txt'
#         np.savetxt(
#             nombre_salida,
#             np.array(datos_extraidos),
#             fmt='%.4f', delimiter='\t',
#             header=header, comments='',
#         )
#         print(f'Datos de {len(datos_extraidos)} simulaciones → {nombre_salida}')
#     else:
#         print('No se encontraron datos para exportar.')


# extraer_datos_simulaciones(carpeta_resultados)